# 09 · H7: reentrenamiento sin EMA (corrección del profesor) + validación + tercera evaluación en test

**Receta H7 = receta de H6 con dos cambios:**

| Opción | H6 | H7 | Por qué |
|---|---|---|---|
| `--ema` | 0.999 | **0** | El profesor pidió no usar EMA (riesgo con imágenes "feas"). |
| `--aug-mm` | no | **sí** | El giro de entrenamiento (±35°) ahora es en milímetros: no deforma la pelvis. |
| Orientación | `--canonical` | `--canonical` | Igual: todos los volúmenes en RAI (cabecera, productos punto y cruz). |

Todo lo demás es idéntico (32 canales, CBAM 1, 60 épocas, cosine, lr 0.002, secundarios ×3, semilla 42): la única diferencia con H6 es la que pidió el profesor.

**Etapas** (cada una guarda en Drive; si Colab se cae, al volver a correr sigue donde iba):
1. Entrenar H7 (≤ 170 min, igual que H6).
2. Validación (15 casos): H6 vs H7, sin giro / giro 60° / giro 60° + enderezado.
3. **Tercera evaluación en test** (15 casos): solo se activa a mano, una vez, con la configuración ya decidida en validación.

Recomendado: GPU **L4 o A100** si hay (T4 también sirve, más lento).

**Dónde correrlo:** necesita **≥ 20 GB de RAM**: **Kaggle** (30 GB, recomendado), Colab Pro con RAM alta, o el entorno donde se entrenó H6. En Colab gratis (12 GB) falla.

**Cómo correrlo:** *Entorno de ejecución → Ejecutar todo*. Las celdas van en orden y cada una espera a la anterior; la de test no hace nada hasta que la activen a mano. Si Colab se reinicia, vuelvan a *Ejecutar todo*: reanuda desde lo guardado en Drive.

In [1]:
!nvidia-smi -L || echo "SIN GPU: activa la GPU antes de seguir"

GPU 0: Tesla T4 (UUID: GPU-c68e8e7e-e17e-dce7-5e57-8b1b69e2078f)
GPU 1: Tesla T4 (UUID: GPU-b0c242cf-4c57-6825-58e8-f9b48a441473)


## 1. Configuración

**Kaggle:** crear un *Dataset* privado llamado **`pengwin-h7`** con `codigo_pengwin.zip`, `cache_s10_todo.zip` y
`h6_auggeo_rot35_canon_ema_w32_e60_s42.pth`; agregarlo al notebook (*Add Input*); *Settings → Accelerator → GPU*.

**Colab (con RAM alta):** esos tres archivos en `Mi unidad/PENGWIN/`.

In [5]:
import os, glob, zipfile, shutil, csv
EN_KAGGLE = True
print("Contenido de /kaggle/input:")
for r in sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]:
    print("  ", r)

def buscar(patron):
    return sorted(glob.glob(f"/kaggle/input/**/{patron}", recursive=True))

CARPETA = "/kaggle/working/resultados"; os.makedirs(CARPETA, exist_ok=True)
H6 = (buscar("h6_auggeo_rot35_canon_ema_w32_e60_s42.pth") or ["NO ENCONTRADO"])[0]
H7 = f"{CARPETA}/h7_auggeo_rot35_canon_aumm_noema_w32_e60_s42.pth"
SEMILLA = 42
os.chdir("/kaggle/working")

# Código: carpeta ya descomprimida o el zip
if not os.path.exists("repo"):
    src = buscar("src/aumento_fisico.py")
    if src:
        shutil.copytree(os.path.dirname(os.path.dirname(src[0])), "repo")
    else:
        z = buscar("codigo_pengwin.zip"); assert z, "No encontré codigo_pengwin.zip ni la carpeta src"
        with zipfile.ZipFile(z[0]) as f: f.extractall("repo_zip")
        shutil.move(glob.glob("repo_zip/*")[0], "repo")

# Caché: carpeta ya descomprimida o el zip
os.chdir("/kaggle/working/repo")
if not os.path.exists("data/cache/s10/cache_384"):
    idx = [p for p in buscar("index.csv") if os.path.isdir(os.path.join(os.path.dirname(p), "cache_384"))]
    os.makedirs("data/cache", exist_ok=True)
    if idx:
        os.symlink(os.path.dirname(idx[0]), "data/cache/s10")
    else:
        z = buscar("cache_s10_todo.zip"); assert z, "No encontré cache_s10_todo.zip ni la carpeta del caché"
        print("Descomprimiendo caché (unos minutos)...")
        with zipfile.ZipFile(z[0]) as f: f.extractall(".")

CD = "data/cache/s10/cache_384"
assert os.path.exists("src/aumento_fisico.py"), "El código no es de la rama feat/normalizacion-orientacion"
splits = [r["split"] for r in csv.DictReader(open("data/cache/s10/index.csv"))]
print("\nH6:", H6)
print("Código:", os.getcwd())
print("Caché:", {s: splits.count(s) for s in set(splits)}, "-> deben ser train 70, val 15, test 15")
!pip -q install SimpleITK scikit-image
!free -h | head -2

Contenido de /kaggle/input:
   /kaggle/input/
   /kaggle/input/datasets
   /kaggle/input/datasets/mariavalencia30
   /kaggle/input/datasets/mariavalencia30/pengwin-h7
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache/s10
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache/s10/cache_384
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache/s10/cache_384/001.npz
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache/s10/cache_384/002.npz
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache/s10/cache_384/003.npz
   /kaggle/input/datasets/mariavalencia30/pengwin-h7/cache_s10_todo/data/cache/s10/cache_384/004.npz
   /kaggle/input/datasets/mariava

## 2. Código y datos (100 casos)

## 3. Entrenar H7

Mismos parámetros que H6 (leídos de su checkpoint) salvo `--ema 0` y `--aug-mm`. El mejor punto se guarda directo en Drive
cada 10 épocas. Si la sesión se cae, al volver a correr reanuda desde el último mejor punto guardado (los pesos, no el optimizador).

In [6]:
# Memoria RAM: el entrenamiento carga ~18 000 cortes (≈ 5.5 GB) en RAM.
# - Kaggle (≈ 30 GB de RAM): sin problema.
# - Colab gratis (≈ 12 GB): falla ("Bus error" o reinicio). Usen Kaggle, Colab Pro (RAM alta) o el entorno donde se entrenó H6.
!free -h | head -2
if not EN_KAGGLE:
    import subprocess
    total_gb = int(subprocess.check_output("free -g | awk 'NR==2{print $2}'", shell=True))
    if total_gb < 20:
        print(f"AVISO: este Colab tiene {total_gb} GB de RAM; el entrenamiento probablemente fallará. Usen Kaggle o Colab con RAM alta.")


               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.3Gi        25Gi       1.2Mi       5.5Gi        30Gi


### Entrenamiento (aquí se ven las épocas)

La primera línea `epoch 001` tarda 5–8 min en una T4. Luego sale una línea por época y, cada 10, una con métricas.
**No detengan la celda.** Termina con `Checkpoint: ... | métricas: ...` (o se detiene sola a los 170 min y guarda el mejor punto).

In [7]:
OPCIONES = ("--cache-root data/cache/s10 --size 384 --mode stretch --epochs 60 --batch-size 16 --lr 0.002 "
            "--base-channels 32 --cbam 1 --instance-mode primary --augment --cosine --margin 10 --neg-fraction 0.15 "
            "--per-epoch 2500 --val-stride 6 --eval-every 10 --max-minutes 170 --secondary-weight 3.0 "
            "--canonical --aug-geo 1.0 --aug-rot-deg 35 --aug-mm --ema 0")
if os.path.exists(H7) and os.path.exists(H7.replace(".pth", ".json")):
    print("H7 ya está entrenado:", H7)
else:
    reanudar = f'--resume "{H7}"' if os.path.exists(H7) else ""
    !free -h | head -2
    !python -u src/train_s10.py {OPCIONES} --seed {SEMILLA} --output "{H7}" {reanudar}

               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.3Gi        25Gi       1.2Mi       5.5Gi        30Gi
Casos: train=70 val=15 | data/cache/s10/cache_384
Cortes train=18266 (con hueso 16876) | val=709 | carga 52s
Modelo: 2,736,958 parámetros | CBAM=1 γ aprendible=False | cuda
epoch 001 loss=5.0455 dice=0.4410 map50=0.1135 inst=0.4091 (152s/época, total 3.5 min)
epoch 002 loss=2.6661 (56s)
epoch 003 loss=2.2204 (57s)
epoch 004 loss=2.0085 (57s)
epoch 005 loss=1.7045 (57s)
epoch 006 loss=1.1181 (57s)
epoch 007 loss=1.0767 (57s)
epoch 008 loss=1.0206 (57s)
epoch 009 loss=0.9976 (57s)
epoch 010 loss=0.9674 dice=0.9211 map50=0.7886 inst=0.7876 (131s/época, total 13.3 min)
epoch 011 loss=0.9509 (57s)
epoch 012 loss=0.9341 (57s)
epoch 013 loss=0.9154 (57s)
epoch 014 loss=0.8892 (57s)
epoch 015 loss=0.8798 (57s)
epoch 016 loss=0.8823 (57s)
epoch 017 loss=0.8568 (57s)
epoch 018 loss=0.8378 (57s)
epoch 019 loss=0.8295 (57s)
epoch 020 loss

In [12]:
import json
info = json.load(open(H7.replace(".pth", ".json")))
print(f"minutos: {info['minutes']:.0f} | cortado por tiempo: {info['stopped_early']}")
print({k: round(v, 3) for k, v in info["best"].items() if k in ("classification_f1_macro", "detection_map50",
      "segmentation_dice_macro", "instance_dice_matched")})

minutos: 67 | cortado por tiempo: False
{'classification_f1_macro': 0.992, 'detection_map50': 0.974, 'segmentation_dice_macro': 0.97, 'instance_dice_matched': 0.875}


## 4. Validación (15 casos): H6 vs H7

Misma prueba del notebook 07. Aquí se decide qué va a test: **H6 o H7**, y **con o sin enderezado**.

In [13]:
CORRIDAS = [("base", ""), ("rotmm60", "--stress rotmm=60"), ("rotmm60_norm", "--stress rotmm=60 --normalizar")]
for etiqueta, ruta in [("H6", H6), ("H7", H7)]:
    for corrida, extra in CORRIDAS:
        salida = f"{CARPETA}/val_{etiqueta}_{corrida}.json"
        if os.path.exists(salida):
            print("ya existe:", salida); continue
        print(f"\n===== {etiqueta} · {corrida} =====")
        !python src/evaluate_volumes.py --checkpoint "{ruta}" --cache-dir {CD} --split val --tta {extra} --output "{salida}" 2>&1 | grep -E "casos de|Guardado|rror"


===== H6 · base =====
15 casos de val | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
^C

===== H6 · rotmm60 =====
15 casos de val | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
^C

===== H6 · rotmm60_norm =====
^C

===== H7 · base =====
^C

===== H7 · rotmm60 =====
^C

===== H7 · rotmm60_norm =====
^C


In [14]:
import pandas as pd
METRICAS = ["classification_f1_macro", "classification_auc_macro", "detection_map50", "detection_iou_mean",
            "segmentation_dice_macro", "instance_dice_matched"]
filas = []
for etiqueta in ("H6", "H7"):
    for corrida, _ in CORRIDAS:
        f = f"{CARPETA}/val_{etiqueta}_{corrida}.json"
        if os.path.exists(f):
            r = json.load(open(f))["subsets"]["todo"]
            filas.append({"modelo": etiqueta, "corrida": corrida, **{m: round(r[m], 3) for m in METRICAS}})
tabla_val = pd.DataFrame(filas).set_index(["modelo", "corrida"])
tabla_val.to_csv(f"{CARPETA}/tabla_validacion_H6_H7.csv")
tabla_val

KeyError: "None of ['modelo', 'corrida'] are in the columns"

### nueva validacion

In [19]:
CORRIDAS = [("base", ""), ("rotmm60", "--stress rotmm=60"), ("rotmm60_norm", "--stress rotmm=60 --normalizar")]
for corrida, extra in CORRIDAS:
    salida = f"{CARPETA}/val_H7_{corrida}.json"
    if os.path.exists(salida):
        print("ya existe:", salida); continue
    print(f"\n===== H7 · {corrida} =====")
    !python src/evaluate_volumes.py --checkpoint "{H7}" --cache-dir {CD} --split val --tta {extra} --output "{salida}" 2>&1 | grep -E "casos de|caso |giro|Guardado|rror"


===== H7 · base =====
15 casos de val | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
  1/15 caso 014: 298 cortes (40 s)
  2/15 caso 019: 350 cortes (87 s)
  3/15 caso 024: 350 cortes (133 s)
  4/15 caso 027: 313 cortes (174 s)
  5/15 caso 033: 365 cortes (223 s)
  6/15 caso 044: 409 cortes (280 s)
  7/15 caso 050: 326 cortes (324 s)
  8/15 caso 057: 341 cortes (369 s)
  9/15 caso 060: 333 cortes (414 s)
  10/15 caso 067: 369 cortes (463 s)
  11/15 caso 077: 350 cortes (509 s)
  12/15 caso 078: 388 cortes (559 s)
  13/15 caso 087: 312 cortes (602 s)
  14/15 caso 095: 257 cortes (636 s)
  15/15 caso 099: 286 cortes (673 s)
Guardado: /kaggle/working/resultados/val_H7_base.json

===== H7 · rotmm60 =====
15 casos de val | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
  1/15 caso 014: 298 cortes (42 s)
  2/15 caso 019: 350 cortes (90 s)
  3/15 caso 024: 350 cortes (136 s)
  4/15 caso 027: 313 cortes (179 s)
  5/15 caso 033: 365 cortes (231 s)
  6

In [20]:
import json, pandas as pd
H6_VAL = {"base":         dict(classification_f1_macro=0.992, detection_map50=0.971, detection_iou_mean=0.949, segmentation_dice_macro=0.971, instance_dice_matched=0.869),
          "rotmm60":      dict(classification_f1_macro=0.959, detection_map50=0.311, detection_iou_mean=0.392, segmentation_dice_macro=0.715, instance_dice_matched=0.608),
          "rotmm60_norm": dict(classification_f1_macro=0.989, detection_map50=0.963, detection_iou_mean=0.928, segmentation_dice_macro=0.957, instance_dice_matched=0.851)}
filas = []
for corrida in ("base", "rotmm60", "rotmm60_norm"):
    filas.append({"modelo": "H6 (EMA)", "corrida": corrida, **H6_VAL[corrida]})
    r = json.load(open(f"{CARPETA}/val_H7_{corrida}.json"))["subsets"]["todo"]
    filas.append({"modelo": "H7 (sin EMA)", "corrida": corrida, **{k: round(r[k], 3) for k in H6_VAL["base"]}})
tabla = pd.DataFrame(filas).set_index(["corrida", "modelo"])
tabla.to_csv(f"{CARPETA}/tabla_validacion_H6_H7.csv"); tabla

classification_f1_macro  detection_map50  \
corrida      modelo                                                   
base         H6 (EMA)                        0.992            0.971   
             H7 (sin EMA)                    0.992            0.974   
rotmm60      H6 (EMA)                        0.959            0.311   
             H7 (sin EMA)                    0.951            0.347   
rotmm60_norm H6 (EMA)                        0.989            0.963   
             H7 (sin EMA)                    0.988            0.963   

                           detection_iou_mean  segmentation_dice_macro  \
corrida      modelo                                                      
base         H6 (EMA)                   0.949                    0.971   
             H7 (sin EMA)               0.948                    0.971   
rotmm60      H6 (EMA)                   0.392                    0.715   
             H7 (sin EMA)               0.426                    0.725   
rotmm60_norm H6 (EMA)                   0.928                    0.957   
             H7 (sin EMA)               0.924                    0.956   

                           instance_dice_matched  
corrida      modelo                               
base         H6 (EMA)                      0.869  
             H7 (sin EMA)                  0.873  
rotmm60      H6 (EMA)                      0.608  
             H7 (sin EMA)                  0.636  
rotmm60_norm H6 (EMA)                      0.851  
             H7 (sin EMA)                  0.855

## 5. Tercera evaluación en test (una sola vez)

**Solo cuando la decisión ya esté tomada en validación** (qué modelo y si va con enderezado). Test se evalúa una vez,
sin prueba de estrés, y se reporta tal cual salga, aunque sea peor. Es la **tercera evaluación** del proyecto
(1.ª: D5 0.845 · 2.ª: H6 0.875).

Para activarla, cambien `CONFIRMO_TERCERA_EVALUACION` a `True` y `MODELO_FINAL` al elegido.

In [23]:
CONFIRMO_TERCERA_EVALUACION = True
MODELO_FINAL = H7            # o H6, según lo que diga la validación

if not CONFIRMO_TERCERA_EVALUACION:
    print("Test NO evaluado. Cuando el equipo decida, cambien CONFIRMO_TERCERA_EVALUACION = True.")
else:
    nombre = os.path.splitext(os.path.basename(MODELO_FINAL))[0]
    for etiqueta, extra in [("sin_enderezar", ""), ("enderezado", "--normalizar")]:
        salida = f"{CARPETA}/test_tercera_{nombre}_{etiqueta}.json"
        if os.path.exists(salida):
            print("ya existe (no se repite test):", salida); continue
        !python src/evaluate_volumes.py --checkpoint "{MODELO_FINAL}" --cache-dir {CD} --split test --tta {extra} --output "{salida}" 2>&1 | grep -E "casos de|AVISO|Guardado|rror"
    filas = []
    for etiqueta in ("sin_enderezar", "enderezado"):
        r = json.load(open(f"{CARPETA}/test_tercera_{nombre}_{etiqueta}.json"))["subsets"]["todo"]
        filas.append({"test": etiqueta, **{m: round(r[m], 3) for m in METRICAS}})
    tabla_test = pd.DataFrame(filas).set_index("test")
    tabla_test.to_csv(f"{CARPETA}/tabla_test_tercera_evaluacion.csv")
    display(tabla_test)

AVISO: test se evalúa una sola vez, con la configuración ya congelada en validación.
15 casos de test | dispositivo cuda | TTA=True | segundo pase=False | normalizar=False
Guardado: /kaggle/working/resultados/test_tercera_h7_auggeo_rot35_canon_aumm_noema_w32_e60_s42_sin_enderezar.json
AVISO: test se evalúa una sola vez, con la configuración ya congelada en validación.
15 casos de test | dispositivo cuda | TTA=True | segundo pase=False | normalizar=True
Guardado: /kaggle/working/resultados/test_tercera_h7_auggeo_rot35_canon_aumm_noema_w32_e60_s42_enderezado.json


,classification_f1_macro,classification_auc_macro,detection_map50,detection_iou_mean,segmentation_dice_macro,instance_dice_matched
test,,,,,,
sin_enderezar,0.989,0.999,0.962,0.940,0.971,0.879
enderezado,0.989,0.999,0.958,0.938,0.968,0.874


In [24]:
r = json.load(open(f"{CARPETA}/val_H7_base.json"))["subsets"]["todo"]
OBJ = {"classification_f1_macro":.85, "classification_auc_macro":.85, "detection_iou_mean":.65, "detection_map50":.65,
       "detection_map50_95":.40, "segmentation_dice_macro":.85, "segmentation_iou_macro":.70,
       "instance_dice_matched":.85, "instance_iou_matched":.70}
for k, o in OBJ.items():
    print(f"{k:28s} {r[k]:.3f}  {'cumple' if r[k] >= o else 'NO cumple'} (objetivo {o})")

classification_f1_macro      0.992  cumple (objetivo 0.85)
classification_auc_macro     0.999  cumple (objetivo 0.85)
detection_iou_mean           0.948  cumple (objetivo 0.65)
detection_map50              0.974  cumple (objetivo 0.65)
detection_map50_95           0.903  cumple (objetivo 0.4)
segmentation_dice_macro      0.971  cumple (objetivo 0.85)
segmentation_iou_macro       0.944  cumple (objetivo 0.7)
instance_dice_matched        0.873  cumple (objetivo 0.85)
instance_iou_matched         0.812  cumple (objetivo 0.7)
